In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

# For LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator


In [2]:
df = pd.read_csv("gage02361000_NADA_all_cells.csv")

print(df.head())
print(df.info())


   year  flow_cfs  cell_1  cell_2  cell_3  cell_4  cell_5  cell_6  cell_7  \
0   365       NaN     NaN     NaN  -2.485  -2.394     NaN     NaN  -2.678   
1   366       NaN     NaN     NaN   0.178  -0.305     NaN     NaN  -0.116   
2   367       NaN     NaN     NaN   0.650   0.233     NaN     NaN   0.384   
3   368       NaN     NaN     NaN  -0.316  -0.337     NaN     NaN  -0.329   
4   369       NaN     NaN     NaN  -0.469  -0.510     NaN     NaN  -0.466   

   cell_8  cell_9  cell_10  cell_11  cell_12  cell_13  
0  -2.870     NaN      NaN   -3.204   -3.113      NaN  
1  -0.393     NaN      NaN   -0.586   -0.672      NaN  
2   0.323     NaN      NaN    0.049    0.114      NaN  
3  -0.305     NaN      NaN   -0.092   -0.121      NaN  
4  -0.468     NaN      NaN   -0.206   -0.258      NaN  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1642 entries, 0 to 1641
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   year      1

In [3]:
# Identify predictor columns
X = df[[c for c in df.columns if c.startswith("cell_")]]
y = df["flow_cfs"]

# Keep years for plotting
years = df["year"]


In [4]:
# Calibration: rows where flow is observed (non-NaN)
calibration_mask = ~y.isna()
X_cal = X[calibration_mask]
y_cal = y[calibration_mask]
years_cal = years[calibration_mask]

# Full reconstruction predictors (for later prediction)
X_full = X
years_full = years


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cal, y_cal, test_size=0.2, random_state=42
)


In [6]:
rf = RandomForestRegressor(n_estimators=500, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"Random Forest – R² = {r2:.3f}, RMSE = {rmse:.3f}")


Random Forest – R² = -0.096, RMSE = 349.720


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [7]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
print("GB R²:", r2_score(y_test, gb.predict(X_test)))

xg = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=42
)
xg.fit(X_train, y_train)
print("XGB R²:", r2_score(y_test, xg.predict(X_test)))


GB R²: 0.07162295405096697
XGB R²: 0.01861149253561878


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPRegressor(hidden_layer_sizes=(64,32), activation="relu",
                   solver="adam", max_iter=500, random_state=42)
mlp.fit(X_train_scaled, y_train)
print("MLP R²:", r2_score(y_test, mlp.predict(X_test_scaled)))


MLP R²: -0.821599742204864


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [9]:
models = {
    "RandomForest": rf,
    "GradientBoosting": gb,
    "XGBoost": xg,
    "MLP": mlp
}

results = []
for name, m in models.items():
    yhat = m.predict(X_test)
    results.append({
        "Model": name,
        "R2": r2_score(y_test, yhat),
        "RMSE": mean_squared_error(y_test, yhat, squared=False)
    })
pd.DataFrame(results)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:486:

,Model,R2,RMSE
0,RandomForest,-0.095676,349.720206
1,GradientBoosting,0.071623,321.915448
2,XGBoost,0.018611,330.978746
3,MLP,-2.128186,590.916238


In [ ]:
full_pred = rf.predict(X_full)
df["recon_flow_rf"] = full_pred

plt.figure(figsize=(10,4))
plt.plot(years_full, full_pred, label="RF Reconstruction", color="blue")
plt.scatter(years_cal, y_cal, color="red", label="Observed Flow")
plt.legend()
plt.title("Random Forest Paleostreamflow Reconstruction")
plt.xlabel("Year"); plt.ylabel("Flow (cfs)")
plt.show()


In [ ]:
df_out = df[["year", "flow_cfs", "recon_flow_rf"]]
df_out.to_csv("gage02361000_RF_reconstruction.csv", index=False)
print("Saved:", "gage02361000_RF_reconstruction.csv")
